# 🐶 Classification of Malignant Melanoma in Canines (CATCH)
### Weeks 1–6 — Segmentation & Classification | Project DAIM2025A_088

**Student:** Muhammad Tayyab Abbas &nbsp;·&nbsp; **Supervisor:** Dr Claire Cashmore &nbsp;·&nbsp; University of Hull

This notebook is the single place that runs and explains the whole project so far:
preprocessing (W1–2), U-Net segmentation (W3–4), the Attention U-Net variant (W5),
and tumour-subtype classification (W6).

## 0. Plan for the classification section — answering the supervisor's questions

> *"Plan the classification part so you are prepared with the segmentation output…
> What is the output of the segmentation? What are you going to classify? How many
> classes? Are there enough of each class? What about balancing — before or after
> the segmentation?"*

**The two-stage pipeline (how segmentation feeds classification):**

```
Whole-slide image
   └─► tile into 256×256 patches (5×/10×/20×)
        └─► [STAGE 1] U-Net SEGMENTATION  ──► per-pixel tumour MASK (tumour vs background)
             └─► keep the patches whose mask says "tumour"  (tumour localisation)
                  └─► [STAGE 2] CNN CLASSIFICATION  ──► tumour SUBTYPE label
```

**Q1 — What is the output of the segmentation?**
A per-pixel **binary mask** for each patch: `1 = tumour`, `0 = background/normal tissue`.
Aggregating the patch masks gives a tumour-region map for the whole slide. This
output is used two ways: (a) to *locate* the tumour, and (b) to *select* the
tumour-containing patches that are then passed to the classifier.

**Q2 — What are we going to classify, and how many classes?**
We classify the **tumour subtype** of each tumour patch. The real CATCH dataset
contains **7 tumour subtypes** — *Melanoma, Mast cell tumour, Squamous cell
carcinoma, Peripheral nerve sheath tumour, Trichoblastoma, Histiocytoma,
Plasmacytoma*. Because the project's focus is **melanoma**, the classifier can run
in two modes: (i) full **7-class** subtype classification, or (ii) **melanoma vs.
non-melanoma** (binary). The demo below uses **3 representative classes**
(melanocytic, mast cell, squamous cell) to validate the pipeline.

**Q3 — Are there enough of each class? (balance)**
CATCH is **imbalanced** — some subtypes have many more slides/patches than others.
We measure the class counts (cell below) and report them so the imbalance is
explicit.

**Q4 — Balancing: before or after the segmentation?**
There are **two different imbalances**, handled at **two different stages**:
- *Segmentation imbalance* (tumour pixels ≈ 9 % vs background) → handled **inside
  segmentation** with a **BCE `pos_weight` + Dice loss**.
- *Class imbalance across subtypes* → handled **after segmentation, at the
  classification stage**, with **class-weighted cross-entropy**, **augmentation**,
  and a **stratified slide-level split**. (A `WeightedRandomSampler` is an
  alternative.)

So: balancing for classification is done **after** segmentation, on the tumour
patches that are fed to the classifier.

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

# find the project root (folder that contains config/config.yaml) and work from there
root = Path.cwd()
while not (root / "config" / "config.yaml").exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
print("Project root:", root)

import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.utils.config import load_config
cfg = load_config()
LOGS = root / "outputs" / "logs"; FIG = root / "outputs" / "figures"

def show_img(path, title=None, size=(6,6)):
    p = Path(path)
    if not p.exists():
        print("(not found yet — run the pipeline to create:", p.name, ")"); return
    import matplotlib.image as mpimg
    plt.figure(figsize=size); plt.imshow(mpimg.imread(str(p))); plt.axis("off")
    if title: plt.title(title)
    plt.show()

def load_json(name):
    p = LOGS / name
    return json.loads(p.read_text()) if p.exists() else None
print("Setup done.")

## 2. Dataset & scale — *"are you using the full dataset?"*

**Honest status:** the real **CATCH dataset (750 whole-slide images)** is hosted on
TCIA and is being downloaded. So that development never stalls, the pipeline is
**validated on synthetic slides** that exercise every step end-to-end. **The exact
same code runs unchanged on the full 750-slide dataset** — only the input folder
changes (see Section 6).

The numbers you may have seen (e.g. *616*) are **extracted patches**, not slides:
one slide produces many 256×256 patches at 5×/10×/20×. The cell below shows the
current patch counts for segmentation and classification.

In [ ]:
# Segmentation dataset size
seg_sum = json.loads((root/"data/processed/seg_splits/seg_split_summary.json").read_text()) \
    if (root/"data/processed/seg_splits/seg_split_summary.json").exists() else None
if seg_sum:
    print("SEGMENTATION patches:", seg_sum["patches"])
    print("  magnifications:", seg_sum.get("magnifications"))

# Classification dataset size + class balance
cls_sum = json.loads((root/"data/processed/cls_splits/cls_split_summary.json").read_text()) \
    if (root/"data/processed/cls_splits/cls_split_summary.json").exists() else None
if cls_sum:
    print("\nCLASSIFICATION patches:", cls_sum["patches"])
    print("  classes:", cls_sum["classes"])
    dist = cls_sum["class_distribution"]
    print("  class distribution (imbalance):", dist)
    plt.figure(figsize=(5,3)); plt.bar(dist.keys(), dist.values(), color="#8E44AD")
    plt.title("Class distribution (patches)"); plt.ylabel("patches"); plt.xticks(rotation=20)
    plt.tight_layout(); plt.show()

## 3. Weeks 1–2 — Preprocessing (live demo)

Quality check → **Macenko stain normalisation** → **tissue detection** →
**multi-magnification patch extraction** → stratified split. Below we run the key
steps live on one synthetic slide.

In [ ]:
from src.utils.demo_segmentation import generate_he_slide_with_mask
from src.preprocessing.stain_normalization import MacenkoNormalizer
from src.preprocessing.tissue import tissue_mask
from src.preprocessing.patch_extraction import extract_multimag_with_mask

img, mask = generate_he_slide_with_mask(size=1024, n_tumours=2, seed=7)
norm = MacenkoNormalizer(**cfg["stain_normalization"]).normalize(img)
tm = tissue_mask(img)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(img);  ax[0].set_title("1. Original H&E");      ax[0].axis("off")
ax[1].imshow(norm); ax[1].set_title("2. Macenko normalised"); ax[1].axis("off")
ax[2].imshow(tm, cmap="gray"); ax[2].set_title("3. Tissue mask"); ax[2].axis("off")
ax[3].imshow(mask, cmap="gray"); ax[3].set_title("4. Tumour mask (ground truth)"); ax[3].axis("off")
plt.tight_layout(); plt.show()

# multi-magnification patch counts
counts = {}
for (m, y, x, p, mp) in extract_multimag_with_mask(norm, mask, [5,10,20], 20,
        256, 256, 0.5, 2000, tissue_img=img):
    counts[m] = counts.get(m, 0) + 1
print("patches per magnification from this slide:", counts)

## 4. Weeks 3–4 — U-Net segmentation (baseline vs ResNet-34)

Two U-Nets were trained: a from-scratch **baseline** and a **ResNet-34 encoder**
(transfer learning). Results below are loaded from the saved reports.

In [ ]:
cmp34 = load_json("week3_4_comparison.json")
if cmp34:
    df = pd.DataFrame(cmp34).T[["test_dice","test_iou","params_million"]]
    display(df.round(3))
show_img(FIG/"unet_resnet34_predictions.png", "ResNet-34 U-Net — test predictions (image | truth | prediction)", (9,9))
show_img(FIG/"unet_resnet34_curves.png", "ResNet-34 U-Net — training curves", (7,4))

## 5. Week 5 — Attention U-Net variant (3-way comparison)

The **Attention U-Net** adds attention gates to the skip connections, focusing on
tumour regions. Built on the baseline backbone so the comparison isolates the
attention effect.

In [ ]:
seg3 = load_json("week5_6_seg_comparison.json")
if seg3:
    df = pd.DataFrame(seg3).T
    display(df.round(3))
else:
    print("Run scripts/run_week5_6_pipeline.py to populate the 3-way comparison.")
show_img(FIG/"unet_attention_predictions.png", "Attention U-Net — test predictions", (9,9))

## 6. Week 6 — Tumour-subtype classification (ResNet-50)

The **classification stage** of the plan from Section 0. A **ResNet-50** (ImageNet
transfer learning, progressive unfreezing, **class-weighted** cross-entropy + label
smoothing) classifies tumour patches into subtypes. Evaluated with accuracy,
macro-F1, AUC-ROC and a confusion matrix.

In [ ]:
clsrep = load_json("cls_resnet50_test_report.json")
if clsrep:
    m = clsrep["test_metrics"]
    print("Classes:", clsrep["classes"])
    print("TEST  accuracy=%.3f  macro-F1=%.3f  precision=%.3f  recall=%.3f  AUC-ROC=%s" % (
        m["accuracy"], m["f1"], m["precision"], m["recall"],
        "n/a" if m["auc_roc"]!=m["auc_roc"] else round(m["auc_roc"],3)))
else:
    print("Run scripts/run_week5_6_pipeline.py to train + evaluate the classifier.")
show_img(FIG/"cls_resnet50_confusion.png", "ResNet-50 — confusion matrix (test)", (5,5))

## 7. How to run on the FULL CATCH dataset

The pipeline is **dataset-agnostic** — point it at the real slides instead of the
demo. Once the CATCH slides are downloaded from TCIA:

```text
data/raw/images/   <- whole-slide images   (.svs / .tiff)
data/raw/masks/    <- tumour masks          (segmentation ground truth)
data/raw/cls_images/<label>_*.png   <- subtype-labelled slides (classification)
```

then run **without** `--demo`:

```bash
python scripts/run_week1_2_pipeline.py     --input data/raw          # preprocess all slides
python scripts/prepare_segmentation_data.py --input data/raw          # seg patches+masks
python scripts/train_unet.py --arch unet                              # ResNet-34 U-Net
python scripts/train_unet.py --arch attention                         # Attention U-Net
python scripts/prepare_classification_data.py --input data/raw        # classification patches
python scripts/train_classifier.py --arch resnet50                    # ResNet-50 classifier
```

No code changes are needed — the same modules scale from 8 demo slides to the full
750-slide CATCH collection.

## 8. Summary & next steps (Weeks 7–8)

**Done (W1–6):** preprocessing · baseline + ResNet-34 + **Attention** U-Net
segmentation · **ResNet-50** subtype classification with class balancing · a clear
segmentation→classification plan.

**Next:** EfficientNet-B3 classifier + comparison · run on the **real CATCH**
slides · evaluation phase (5-fold cross-validation, McNemar's test, Grad-CAM).